# RAG-Powered Document Assistant — Harry Potter Edition

**Track:** Core Track (text-only RAG assistant)
**Domain:** Study/chat assistant over the 7 Harry Potter books (~3,679 pages combined)
**Deliverable:** `notebooks/rag_pipeline.ipynb` — cleans & chunks the data, generates embeddings,
stores them in a vector database, and builds + evaluates a RAG pipeline using an LLM.

This notebook follows the graduation-project brief section by section (Phase 2.1 → 2.7).
Each phase starts with a markdown cell explaining *what* the phase does and *why*, followed
by the code cell(s) that implement it, so the notebook reads like a report.

In [1]:
!pip install -q pymupdf sentence-transformers qdrant-client python-dotenv langchain-groq langchain-google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 41.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import os


PDF_PATH = Path("/content/harrypotter.pdf")
MARKDOWN_PATH = Path("output.md")
DATASET_FOLDER = Path("dataset")
DATASET_FOLDER.mkdir(exist_ok=True)


## Phase 2.1 — Load & Inspect

**Domain:** the 7 Harry Potter novels (Sorcerer's/Philosopher's Stone → Deathly Hallows), treated as one
document collection so the assistant can answer questions across the whole series.

**Format:** a single PDF (or 7 PDFs merged into one before this step). Each page is extracted as text
and converted into a per-page Markdown file — this makes it trivial to later split by book and page
number, and to sanity-check that no page silently returned empty text.

The cell below extracts every page's text and reports parsing health: total pages, empty/near-empty
pages (a strong signal of a scanned page that would need OCR — Harry Potter PDFs are almost always
already text-based, but we check anyway), and average characters per page.

In [3]:
import pymupdf


def pdf_to_markdown(pdf_path, markdown_path):
    """Extract text from every page of a PDF and save it as one Markdown file,
    with a '## Page N' header before each page's content so we can split it later."""

    markdown = []
    page_lengths = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text()
            page_lengths.append(len(text.strip()))

            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(text)
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    return page_lengths


page_lengths = pdf_to_markdown(PDF_PATH, MARKDOWN_PATH)

total_pages = len(page_lengths)
empty_pages = [i + 1 for i, n in enumerate(page_lengths) if n < 20]
avg_chars = sum(page_lengths) / total_pages if total_pages else 0

print(f"Total pages extracted: {total_pages}")
print(f"Pages that look empty/near-empty (<20 chars, possibly needs OCR): {len(empty_pages)}")
print(f"First 20 flagged pages: {empty_pages[:20]}")
print(f"Average characters per page: {avg_chars:.0f}")


Total pages extracted: 3623
Pages that look empty/near-empty (<20 chars, possibly needs OCR): 21
First 20 flagged pages: [1, 2, 3, 4, 5, 7, 8, 276, 277, 567, 568, 941, 942, 1209, 1562, 1563, 2408, 2409, 2865, 2966]
Average characters per page: 1732


## Phase 2.2 — Chunking Strategy

**Strategy chosen: page-based chunking, tagged with book title + page number, no additional overlap.**

**Why page-based instead of fixed-size/overlapping chunks:**
- A Harry Potter page is a natural narrative unit (~250–400 words) — small enough to embed precisely,
  large enough to contain a full scene beat, so splitting mid-page would cut sentences and hurt both
  embedding quality and citation clarity.
- Keeping `book_name` + `page_number` as payload gives free, exact citations ("Chamber of Secrets, p. 142")
  without any extra offset bookkeeping that fixed-size chunking would need.
- Overlap is unnecessary here: the loss case fixed-size overlap protects against (a fact split across a
  chunk boundary) is rare within a single page, and we compensate by retrieving `top_k > 1` pages at
  query time so adjacent context is very likely to be pulled in anyway.
- Trade-off acknowledged: very short exchanges of dialogue near a page boundary can still be split across
  two pages; mitigated by retrieving 3–5 pages per query rather than 1.

The code below reads `output.md`, slices it into the 7 books using page-range boundaries, and writes one
small text file per page into `dataset/`, each starting with the book title and page number.

In [5]:
import re

INPUT_FILE = MARKDOWN_PATH
OUTPUT_FOLDER = DATASET_FOLDER

# Update these (start_page, end_page) boundaries to match YOUR merged PDF's actual page numbers —
# they depend on which edition/printing you used to build harry_potter_complete.pdf.
BOOK_RANGES = [
    ("Harry Potter and the Sorcerer's Stone",       12,   274),
    ("Harry Potter and the Chamber of Secrets",     282,  565),
    ("Harry Potter and the Prisoner of Azkaban",    573,  939),
    ("Harry Potter and the Goblet of Fire",         949,  1560),
    ("Harry Potter and the Order of the Phoenix",   1570, 2406),
    ("Harry Potter and the Half-Blood Prince",      2409, 2964),
    ("Harry Potter and the Deathly Hallows",        2967, 3679),
]

raw_text = INPUT_FILE.read_text(encoding="utf-8")

# Split the markdown on our own page headers so each page's raw content is isolated.
page_blocks = re.split(r"(## Page \d+)", raw_text)[1:]
pages_by_number = {}
for header, body in zip(page_blocks[0::2], page_blocks[1::2]):
    page_number = int(header.replace("## Page ", "").strip())
    pages_by_number[page_number] = body.strip()

written = 0
for book_name, start, end in BOOK_RANGES:
    for page_number in range(start, end + 1):
        content = pages_by_number.get(page_number, "").strip()
        if not content:
            continue

        out_path = OUTPUT_FOLDER / f"{book_name.replace(' ', '_')}_p{page_number}.txt"
        out_path.write_text(
            f"# {book_name}\n\n## Page {page_number}\n\n{content}",
            encoding="utf-8",
        )
        written += 1

print(f"Wrote {written} per-page chunk files to {OUTPUT_FOLDER}/")


Wrote 3574 per-page chunk files to dataset/


## Phase 2.3 — Embeddings & Vector Store

Each page-chunk is embedded with a multilingual Sentence-Transformers model (`intfloat/multilingual-e5-large`)
and stored in **Qdrant**, along with `book_name` and `page_number` as payload so retrieval results are
citable without a second lookup.

Qdrant Cloud (free tier) is used here instead of a purely local/on-disk store so the **FastAPI backend can
connect to the same persisted collection over the network without re-running this notebook or shipping a
local vector-store folder** — satisfying "persist the vector store so the backend can load it without
rebuilding" from the brief. If you'd rather ship a local on-disk store, swap `QdrantClient(url=..., api_key=...)`
for `QdrantClient(path="qdrant_data")` and commit/copy that folder into `backend/data/vector_store/`.

In [6]:
from sentence_transformers import SentenceTransformer

DATASET_FOLDER = Path("dataset")
MODEL_NAME = "intfloat/multilingual-e5-large"


def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()
    book_name = lines[0].replace("# ", "").strip()
    page_number = int(lines[2].replace("## Page ", "").strip())
    content = "\n".join(lines[3:]).strip()

    return {"book_name": book_name, "page_number": page_number, "content": content}


page_files = sorted(DATASET_FOLDER.glob("*.txt"))
pages = [read_page(f) for f in page_files]
print(f"Loaded {len(pages)} page-chunks for embedding")

model = SentenceTransformer(MODEL_NAME)

# e5 models expect a "passage: " / "query: " prefix convention.
texts = [f"passage: {p['content']}" for p in pages]
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print(f"Generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}")


Loaded 3574 page-chunks for embedding


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

Generated 3574 embeddings of dimension 1024


In [7]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Foun

In [10]:
print(chromadb.__version__)

1.5.9


In [11]:
import chromadb

CHROMA_PATH = "chroma_data"
COLLECTION_NAME = "harry_potter_pages"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids = [str(i) for i in range(len(pages))]
metadatas = [{"book_name": p["book_name"], "page_number": p["page_number"]} for p in pages]
documents = [p["content"] for p in pages]
embedding_list = [e.tolist() for e in embeddings]

BATCH = 256
for i in range(0, len(pages), BATCH):
    collection.add(
        ids=ids[i : i + BATCH],
        embeddings=embedding_list[i : i + BATCH],
        metadatas=metadatas[i : i + BATCH],
        documents=documents[i : i + BATCH],
    )

print(f"Added {collection.count()} chunks to ChromaDB collection '{COLLECTION_NAME}', persisted at ./{CHROMA_PATH}")


Added 3574 chunks to ChromaDB collection 'harry_potter_pages', persisted at ./chroma_data


## Bonus — Query Routing (optional enhancement, not required by the brief)

Before searching, a small Groq model classifies the incoming message as `retrieve` (needs the books),
`chitchat` (greeting/thanks — answer directly), or `off-topic` (unrelated — decline and redirect). This
avoids running a vector search for messages that don't need one. The required Phase 2.4 `retrieve()` /
`build_prompt()` / `answer()` functions below don't depend on this — you can call them with or without
routing in front.

In [32]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

query = input("Ask a question: ")

router_llm = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

SYSTEM_PROMPT = """You classify messages for a Harry Potter book search system.
Return exactly one label and nothing else:
retrieve - questions about the books, characters, places, or events
chitchat - greetings, thanks, or casual conversation, in this case you can answer the question
off-topic - anything unrelated to the books, and tell the user that you only answer questions about the Harry Potter books"""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]

route = router_llm.invoke(router_messages).content.strip().lower()
route = route.splitlines()[0].strip(" `.,:")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)


Ask a question: Who is Harry Potter?
Route: retrieve


In [35]:
top_k = 3
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-large"
)
if route == "retrieve":
    query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)[0].tolist()

    results = collection.query(query_embeddings=[query_vector], n_results=top_k)

    context = ""
    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        content = results["documents"][0][i]
        score = 1 - results["distances"][0][i]  # cosine space -> similarity

        context += (
            f"Book: {metadata['book_name']}\n"
            f"Page: {metadata['page_number']}\n"
            f"Content: {content}\n\n"
        )

        print("Score:", score)
        print("Book:", metadata["book_name"])
        print("Page:", metadata["page_number"])
        print("Content:", content)
        print("-" * 80)
else:
    print("No database search needed.")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Score: 0.8167393207550049
Book: Harry Potter and the Goblet of Fire
Page: 1459
Content: down the table were sniggering, twisting in their seats to see Harry’s reaction.
“Let me see it,” Harry said to Ron. “Give it here.”
Very reluctantly, Ron handed over the newspaper. Harry turned it over and
found himself staring at his own picture, beneath the banner headline:
HARRY POTTER
“DISTURBED AND DANGEROUS”
The boy who defeated He-Who-Must-Not-Be-Named is unstable
and possibly dangerous, writes Rita Skeeter, Special Correspondent.
Alarming evidence has recently come to light about Harry Potter’s
strange behavior, which casts doubts upon his suitability to compete
in a demanding competition like the Triwizard Tournament, or even
to attend Hogwarts School.
Potter, the Daily Prophet can exclusively reveal, regularly
collapses at school, and is often heard to complain of pain in the
scar on his forehead (relic of the curse with which You-Know-Who
attempted to kill him). On Monday last, midway th

### Bonus — Keyword Search

Semantic search finds similar *meaning* via embeddings. Keyword search looks for the exact words in the
page content instead. For now this searches the `pages` list already loaded in memory; BM25 could be
swapped in later as a stronger keyword-ranking method.

In [36]:
if route == "retrieve":
    keyword_query = "Cedric Diggory"
    top_k = 3

    keywords = keyword_query.lower().split()
    keyword_results = []

    for page in pages:
        content = page["content"].lower()
        score = sum(content.count(keyword) for keyword in keywords)

        if score > 0:
            keyword_results.append({
                "score": score,
                "book_name": page["book_name"],
                "page_number": page["page_number"],
                "content": page["content"],
            })

    keyword_results.sort(key=lambda result: result["score"], reverse=True)

    for result in keyword_results[:top_k]:
        print("Keyword score:", result["score"])
        print("Book:", result["book_name"])
        print("Page:", result["page_number"])
        print("Content:", result["content"])
        print("-" * 80)


Keyword score: 14
Book: Harry Potter and the Goblet of Fire
Page: 1472
Content: and lay motionless, facedown in the grass. Harry dashed over to Cedric, who
had stopped twitching and was lying there panting, his hands over his face.
“Are you all right?” Harry said roughly, grabbing Cedric’s arm.
“Yeah,” panted Cedric. “Yeah . . . I don’t believe it . . . he crept up behind
me. . . . I heard him, I turned around, and he had his wand on me. . . .”
Cedric got up. He was still shaking. He and Harry looked down at Krum.
“I can’t believe this . . . I thought he was all right,” Harry said, staring at
Krum.
“So did I,” said Cedric.
“Did you hear Fleur scream earlier?” said Harry.
“Yeah,” said Cedric. “You don’t think Krum got her too?”
“I don’t know,” said Harry slowly.
“Should we leave him here?” Cedric muttered.
“No,” said Harry. “I reckon we should send up red sparks. Someone’ll
come and collect him . . . otherwise he’ll probably be eaten by a skrewt.”
“He’d deserve it,” Cedric muttered, but

## Phase 2.4 — Retrieval & Prompting

`retrieve(query, top_k)` embeds the question with the same model (using the `query:` prefix that e5
expects) and searches ChromaDB for the closest page vectors. The prompt template then wraps the retrieved
pages as labeled, citable context and instructs the LLM to answer **only** from that context — this is
the "real grounding" the brief explicitly grades on, as opposed to answering from the LLM's own training
knowledge of the books.

We test retrieval against 10 sample questions spanning all 7 books before wiring in generation, so
retrieval quality can be judged independently of the LLM's phrasing.

In [38]:
class Hit:
    """Small wrapper so downstream code can keep using hit.payload / hit.score
    regardless of which vector DB client produced the result."""
    def __init__(self, payload, score):
        self.payload = payload
        self.score = score


def retrieve(query, top_k=5):
    query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True)[0].tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=top_k)

    hits = []
    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        payload = {
            "book_name": metadata["book_name"],
            "page_number": metadata["page_number"],
            "content": results["documents"][0][i],
        }
        # collection was created with cosine space, so distance -> similarity is 1 - distance
        score = 1 - results["distances"][0][i]
        hits.append(Hit(payload=payload, score=score))
    return hits


SAMPLE_QUESTIONS = [
    "How did Harry get the scar on his forehead?",
    "What position does Harry play on the Gryffindor Quidditch team?",
    "Who is the Half-Blood Prince?",
    "What creature guards the entrance to the Chamber of Secrets?",
    "How did Sirius Black escape from Azkaban?",
    "What is the name of Hagrid's giant spider?",
    "Who betrayed Harry's parents to Voldemort?",
    "What does the Marauder's Map show?",
    "How did Dobby the house-elf gain his freedom?",
    "What are the three Deathly Hallows?",
]

for q in SAMPLE_QUESTIONS:
    hits = retrieve(q, top_k=3)
    top = hits[0].payload if hits else None
    print(f"Q: {q}")
    if top:
        print(f"  -> top hit: {top['book_name']}, p.{top['page_number']} (score={hits[0].score:.3f})")
    else:
        print("  -> no hits")


Q: How did Harry get the scar on his forehead?
  -> top hit: Harry Potter and the Goblet of Fire, p.961 (score=0.848)
Q: What position does Harry play on the Gryffindor Quidditch team?
  -> top hit: Harry Potter and the Prisoner of Azkaban, p.791 (score=0.832)
Q: Who is the Half-Blood Prince?
  -> top hit: Harry Potter and the Half-Blood Prince, p.2868 (score=0.814)
Q: What creature guards the entrance to the Chamber of Secrets?
  -> top hit: Harry Potter and the Chamber of Secrets, p.408 (score=0.806)
Q: How did Sirius Black escape from Azkaban?
  -> top hit: Harry Potter and the Order of the Phoenix, p.1753 (score=0.844)
Q: What is the name of Hagrid's giant spider?
  -> top hit: Harry Potter and the Deathly Hallows, p.3528 (score=0.852)
Q: Who betrayed Harry's parents to Voldemort?
  -> top hit: Harry Potter and the Goblet of Fire, p.1488 (score=0.825)
Q: What does the Marauder's Map show?
  -> top hit: Harry Potter and the Goblet of Fire, p.1344 (score=0.803)
Q: How did Dobby the h

In [39]:
def build_prompt(query, retrieved_pages):
    context_blocks = []
    for r in retrieved_pages:
        p = r.payload
        context_blocks.append(f"[{p['book_name']}, p.{p['page_number']}]\n{p['content']}")

    context = "\n\n---\n\n".join(context_blocks)

    system_prompt = (
        "You are a Harry Potter book assistant. Answer the user's question using ONLY the "
        "context pages below. Every claim in your answer must be traceable to a page in the "
        "context. Cite the book title and page number for each fact you use, like "
        "(Chamber of Secrets, p.142). If the context does not contain the answer, say you "
        "don't know rather than guessing."
    )

    user_prompt = f"Context pages:\n\n{context}\n\nQuestion: {query}"
    return system_prompt, user_prompt


def answer(query, top_k=5, llm=None):
    """Full retrieve -> prompt -> generate step. `llm` is any LangChain chat model
    (ChatGroq, ChatGoogleGenerativeAI, etc.) — see Phase 2.6 for a wired example."""
    hits = retrieve(query, top_k=top_k)
    system_prompt, user_prompt = build_prompt(query, hits)
    sources = [f"{r.payload['book_name']}, p.{r.payload['page_number']}" for r in hits]

    if llm is None:
        return {"answer": None, "sources": sources, "system_prompt": system_prompt, "user_prompt": user_prompt}

    from langchain_core.messages import SystemMessage, HumanMessage
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
    return {"answer": response.content, "sources": sources}


## Phase 2.5 — Vision Component

**Not applicable — Core Track was chosen for this project** (text-only RAG assistant over the 7 book
PDFs). No image dataset or YOLO/CV component is included. If this were extended to the Extended Track,
scanned illustration pages or fan-art/diagram datasets could be run through a pretrained YOLO model and
the detected labels appended to the retrieved-page context before generation.

## Phase 2.6 — Evaluation

Two complementary checks, as required by the brief:

1. **Retrieval precision/recall** against hand-labeled "ground truth" pages for a handful of questions
   where we know exactly which page(s) contain the answer.
2. **LLM-as-a-judge** over 10 test questions: an LLM answers each question from the retrieved context,
   then a judge call checks whether the answer is correct and grounded (not hallucinated), producing the
   results table the brief asks for.

In [40]:
# 1) Precision / Recall against known ground-truth pages.
# Update these page numbers to match YOUR merged PDF's actual pagination.
evaluation_cases = [
    {"query": "Who rescued Harry from his locked bedroom using a flying car?", "relevant_pages": {301, 302}},
    {"query": "What loophole did Mr Weasley write into the law about enchanting a car?", "relevant_pages": {314}},
]

top_k = 3
precision_scores, recall_scores = [], []

for case in evaluation_cases:
    hits = retrieve(case["query"], top_k=top_k)
    retrieved_pages = {r.payload["page_number"] for r in hits}
    relevant_retrieved = retrieved_pages & case["relevant_pages"]

    precision = len(relevant_retrieved) / len(retrieved_pages) if retrieved_pages else 0
    recall = len(relevant_retrieved) / len(case["relevant_pages"]) if case["relevant_pages"] else 0

    precision_scores.append(precision)
    recall_scores.append(recall)
    print(f"Q: {case['query']}")
    print(f"  retrieved pages: {sorted(retrieved_pages)} | relevant: {case['relevant_pages']}")
    print(f"  precision={precision:.2f} recall={recall:.2f}\n")

print(f"Mean precision: {sum(precision_scores)/len(precision_scores):.2f}")
print(f"Mean recall: {sum(recall_scores)/len(recall_scores):.2f}")


Q: Who rescued Harry from his locked bedroom using a flying car?
  retrieved pages: [302, 303, 967] | relevant: {301, 302}
  precision=0.33 recall=0.50

Q: What loophole did Mr Weasley write into the law about enchanting a car?
  retrieved pages: [314, 467, 2056] | relevant: {314}
  precision=0.33 recall=1.00

Mean precision: 0.33
Mean recall: 0.75


In [49]:
# 2) LLM-as-a-judge over 10 questions -> results table (question / source / answer / correct?)
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

gemini_llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_JUDGE_MODEL"),
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

TEST_QUESTIONS = SAMPLE_QUESTIONS  # reuse the 10 questions from Phase 2.4

results_table = []
for q in TEST_QUESTIONS:
    result = answer(q, top_k=3, llm=gemini_llm)

    judge_prompt = (
        "You are grading a RAG answer for a Harry Potter book assistant.\n"
        f"Question: {q}\n"
        f"Answer given: {result['answer']}\n"
        f"Sources used: {result['sources']}\n\n"
        "Reply with exactly one word: CORRECT if the answer is accurate and clearly grounded "
        "in the sources, or INCORRECT if it looks hallucinated, wrong, or unsupported."
    )
    judge_response = gemini_llm.invoke(
    [HumanMessage(content=judge_prompt)]
)

verdict = judge_response.content

if isinstance(verdict, list):
    verdict = "".join(
        item.get("text", "") if isinstance(item, dict) else str(item)
        for item in verdict
    )

verdict = verdict.strip().upper()

results_table.append({
    "question": q,
    "retrieved_source": "; ".join(result["sources"]),
    "answer": result["answer"],
    "correct": verdict,
})

import pandas as pd
results_df = pd.DataFrame(results_table)
results_df


,question,retrieved_source,answer,correct
0,What are the three Deathly Hallows?,"Harry Potter and the Deathly Hallows, p.3321; ...","[{'type': 'text', 'text': 'The three Deathly H...",CORRECT


In [50]:
import json

export_config = {
    "embedding_model": MODEL_NAME,
    "vector_db": "chromadb",
    "chroma_path": CHROMA_PATH,
    "collection_name": COLLECTION_NAME,
    "chunking_strategy": "page-based (one page = one chunk), tagged with book_name + page_number",
    "num_chunks_indexed": len(pages),
    "top_k_default": 5,
}

Path("vector_store_config.json").write_text(json.dumps(export_config, indent=2), encoding="utf-8")
print("Wrote vector_store_config.json — copy this into backend/data/vector_store/ alongside your .env.example")
print(json.dumps(export_config, indent=2))


Wrote vector_store_config.json — copy this into backend/data/vector_store/ alongside your .env.example
{
  "embedding_model": "intfloat/multilingual-e5-large",
  "vector_db": "chromadb",
  "chroma_path": "chroma_data",
  "collection_name": "harry_potter_pages",
  "chunking_strategy": "page-based (one page = one chunk), tagged with book_name + page_number",
  "num_chunks_indexed": 3574,
  "top_k_default": 5
}


In [51]:
from pathlib import Path

EXPORT_DIR = Path("rag_export")
EXPORT_DIR.mkdir(exist_ok=True)

print("Created:", EXPORT_DIR)

Created: rag_export


In [53]:
import shutil
from google.colab import files

# Compress the whole export folder
shutil.make_archive("rag_export", "zip", "rag_export")

# Download to your computer
files.download("rag_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>